In [ ]:
!pip install sympy
!pip install networkx

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import numpy as np
import networkx as nx

from itertools import permutations
from sympy.combinatorics.permutations import Permutation

import sys 
sys.path.append("../lecture6/")
import simplicial
import simplicial.drawing
from simplicialx.simplicial import SimplicialComplex

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

from tqdm import tqdm

np.set_printoptions(precision=2, suppress=True)

In [ ]:
def neighbors_lower(x_id, A):
    LT = np.tril(A)
    return list(np.nonzero(LT[x_id])[0])

def intersect(d):
    return list(set(d[0]).intersection(*d))

def grade(K):
    dim = len(max(K, key=len))
    K_graded = [[] for _ in range(dim)]
    for sigma in K:
        dim_sigma = len(sigma) - 1
        #print("{}-dim simplex: {}".format(dim_sigma, sigma))
        if dim_sigma == 0:
            sigma = sigma[0]
        K_graded[dim_sigma].append(sigma)
        
    for k, item in enumerate(K_graded):
        if k==0:
            item_array = np.expand_dims(np.array(item), 1)
        else:
            item_array = np.array(item)
        
        K_graded[k] = item_array

    return K_graded

def expand_incremental(G, k=2):
    
    k = k+1 # k-clique instead of (k-1)-simplex
    
    def add_cofaces(A, k, tau, N_lower, simplices):
        
        # V = V \cup tau
        simplices.append(tau)
        
        # if dim(tau) >= k
        if len(tau) >= k:
            return
        else:
            # foreach v \in N
            for v in N_lower:
                
                # \sigma = \tau \cup v
                sigma = sorted(tau + [v])
                M = intersect([N_lower, neighbors_lower(v, A)])
                add_cofaces(A, k, sigma, M, simplices)
            
        
        return simplices
    
    simplices = []
    A = nx.to_numpy_array(G)

    # define vertex set
    V = [item for item in nx.nodes(G)]
    
    for u in V:
        N_lower = neighbors_lower(u, A)
        add_cofaces(A, k, [u], N_lower, simplices)
    
    return grade(simplices)

def boundary_matrix(K, k=1):
    
    # get simplices, lists of arrays
    sigmas, taus = K[k], K[k-1]
    
    # init boundary matrix B
    B = np.zeros((len(taus), len(sigmas)))
    
    # iterate over columns of B, i.e. k-simplices sigmas
    for j, sigma in enumerate(sigmas):
        
        # omit h-th vertex in a k-simplex sigma
        for h in range(k+1):
            
            idx_h = np.ones(k+1).astype(bool)
            idx_h[h] = False
            
            # get k-1 simplex tau and find i of it
            tau = sigma[idx_h]
            i = np.flatnonzero(np.equal(taus, tau).all(axis=1))[0]
            
            B[i,j] = (-1) ** h # orientation sign
            
    return B

def generalized_boundary_matrix(K, k=1, p=1, eta_sign=1):
    
    # get simplices, lists of arrays
    sigmas, taus = K[k], K[k-p]
    
    # init boundary matrix B
    B = np.zeros((len(taus), len(sigmas)))
    
    # iterate over columns of B, i.e. k-simplices sigmas
    for j, sigma in enumerate(sigmas):
        
        mask = np.ones(k+1).astype(bool)
        mask[:p] = False

        # omit h_1,...,h_j vertices in a k-simplex sigma
        for idx in np.array(sorted(set(permutations(mask)))):

            # get k-p simplex tau and find i of it
            tau = sigma[idx]
            i = np.flatnonzero(np.equal(taus, tau).all(axis=1))[0]

            vertices_in = list(np.array(range(k+1))[idx])
            vertices_out = list(np.array(range(k+1))[~idx])

            B[i,j] = eta_sign * Permutation(vertices_out + vertices_in).signature() # orientation sign
            
    return B

def normalize(L):
    n = L.shape[0]
    D = np.diag(np.diag(L)) + np.eye(n) * 1e-9
    D_inv = np.linalg.inv(D)
    return D_inv @ L

In [ ]:
def orient(B):
    
    BB = np.zeros_like(B).astype(int)

    for j, column_ in enumerate(B.T):
        column = np.copy(column_)
        nonzero_idx, = np.nonzero(column)
        column[nonzero_idx[-2]] = -1
        BB[:,j] = column
        
    return BB


def schur_blocks(M, A_row_idx, B_col_idx):
    
    A_row_idx = np.array(A_row_idx)
    A_col_idx = np.array(B_col_idx)

    D_row_idx = np.setdiff1d(np.arange(len(M)), A_row_idx)
    D_col_idx = np.setdiff1d(np.arange(len(M)), A_col_idx)

    A_sub = M[np.ix_(A_row_idx, A_col_idx)]
    D_sub = M[np.ix_(D_row_idx, D_col_idx)]
    B_sub = M[np.ix_(A_row_idx, D_col_idx)]

    return A_sub, B_sub, D_sub

def schur_complement(M, A_row_idx, B_col_idx):
    A, B, D = schur_blocks(M, A_row_idx, B_col_idx)
    return A - B @ pinv(D) @ B.T

# Seminar 7: Higher-order Laplacian

## Graph Laplacian

Consider the graph $G$. Given its adjacency and incidence matrices arrive to the Laplacian matrix built by both adjacency and incidence approach. Check whether the Laplacian matrices are the same. Compute the spectrum of the graph Laplacian, what can be said of the topology of the graph $G$ based on the computed spectrum?

In [ ]:
# create graph
cmplx = simplicial.SimplicialComplex()

# add 0-simplices (vertices)
v1 = cmplx.addSimplex(id="v1")
v2 = cmplx.addSimplex(id="v2")
v3 = cmplx.addSimplex(id="v3")
v4 = cmplx.addSimplex(id="v4")
v5 = cmplx.addSimplex(id="v5")
v6 = cmplx.addSimplex(id="v6")
v7 = cmplx.addSimplex(id="v7")

# add 1-simplices (edges)
cmplx.addSimplex(['v2', 'v3'], id="e1")
cmplx.addSimplex(['v4', 'v5'], id="e2")
cmplx.addSimplex(['v4', 'v6'], id="e3")
cmplx.addSimplex(['v5', 'v6'], id="e4")
cmplx.addSimplex(['v5', 'v7'], id="e5")
cmplx.addSimplex(['v6', 'v7'], id="e6")

In [ ]:
# set coordinates for vertices
em = simplicial.Embedding(cmplx)
em.positionSimplex(v1, (0.0, 0.5))

em.positionSimplex(v2, (0.25, 1.0))
em.positionSimplex(v3, (0.25, 0.0))

em.positionSimplex(v4, (1.0, 1.0))
em.positionSimplex(v5, (0.5, 0.66))
em.positionSimplex(v6, (1.0, 0.33))
em.positionSimplex(v7, (0.5, 0.0))

# draw simplicial complex
fig = plt.figure(figsize=(6,6))
plt.title("Geometric realization of a graph $G$")
simplicial.drawing.draw_complex(cmplx, em)

#### Adjacency matrix

In [ ]:
A = np.array([
    [0, 0, 0, 0, 0, 0, 0], # v1
    [0, 0, 0, 1, 0, 0, 0], # v2
    [0, 0, 0, 0, 1, 1, 1], # v4
    [0, 1, 0, 0, 0, 0, 0], # v3
    [0, 0, 1, 0, 0, 1, 1], # v5
    [0, 0, 1, 0, 1, 0, 0], # v6
    [0, 0, 1, 0, 1, 0, 0], # v7
])

A

#### Degree matrix

Compute the degree matrix $\mathbf{D}$ based on the adjacency matrix.

In [ ]:
D = np.diag(np.sum(A, axis=0))
D

#### Incidence matrix

In [ ]:
B = orient(cmplx.boundaryMatrix(k=1))
B

#### Laplacians

In [ ]:
LA = D - A
LB = B @ B.T

LB == LA

In [ ]:
LA, np.linalg.eigvalsh(LA)

In [ ]:
LB, np.linalg.eigvalsh(LB)

## Higher-order Laplacian

### Graph

In [ ]:
# draw simplicial complex
fig = plt.figure(figsize=(6,6))
plt.title("Geometric realization of a graph $G$")
simplicial.drawing.draw_complex(cmplx, em)

#### Edge-vertex boundary matrix

In [ ]:
B1 = orient(cmplx.boundaryMatrix(k=1))
B1

#### Laplacian $L_0$

The lower part of the $L_0$ Laplacian is zero as $\partial_0: C_0 \rightarrow 0$ is a zero map.

\begin{align}
\mathbf{L}_k &= \mathbf{B}_k^T \mathbf{B}_k + \mathbf{B}_{k+1} \mathbf{B}_{k+1}^T\\
\mathbf{L}_0 &= \color{red}{\mathbf{B}_0^T \mathbf{B}_0} + \mathbf{B}_1 \mathbf{B}_1^T
\end{align}

In [ ]:
L0 = B1 @ B1.T
L0, np.linalg.eigvalsh(L0)

#### Laplacian $L_1$

The upper part of the $L_1$ Laplacian of a graph (a simplicial complex of dimension one) is zero as $\partial_2: C_2 \rightarrow C_1$ is a zero map, as $C_2$ is empty.

\begin{align}
\mathbf{L}_k &= \mathbf{B}_k^T \mathbf{B}_k + \mathbf{B}_{k+1} \mathbf{B}_{k+1}^T\\
\mathbf{L}_1 &= \mathbf{B}_1^T \mathbf{B}_1 + \color{red}{\mathbf{B}_2 \mathbf{B}_2^T}
\end{align}

In [ ]:
L1 = B1.T @ B1
L1, np.linalg.eigvalsh(L1)

### Simplicial complex

In [ ]:
# add 2-simplex (triangle)
cmplx.addSimplex(['e2', 'e3', 'e4'], id="t1")

In [ ]:
# draw simplicial complex
fig = plt.figure(figsize=(6,6))
plt.title("Geometric realization of a simplicial complex $K$")
simplicial.drawing.draw_complex(cmplx, em)

#### Edge-vertex boundary matrix

In [ ]:
B1 = orient(cmplx.boundaryMatrix(k=1))
B1

#### Triangle-edge boundary matrix

In [ ]:
B2 = orient(cmplx.boundaryMatrix(k=2))
B2

#### Laplacian $L_1$

In general the Laplacian matrix is the sum of the lower and upper parts.

\begin{align}
\mathbf{L}_k &= \mathbf{B}_k^T \mathbf{B}_k + \mathbf{B}_{k+1} \mathbf{B}_{k+1}^T\\
\mathbf{L}_1 &= \mathbf{B}_1^T \mathbf{B}_1 + \mathbf{B}_2 \mathbf{B}_2^T
\end{align}

#### Down Laplacian $L_1$

In [ ]:
L1_down = B1.T @ B1
L1_down, np.linalg.eigvalsh(L1_down)

#### Up Laplacian $L_1$

In [ ]:
L1_up = B2 @ B2.T
L1_up, np.linalg.eigvalsh(L1_up)

#### Laplacian $L_1$

In [ ]:
L1 = B1.T @ B1 + B2 @ B2.T
L1, np.linalg.eigvalsh(L1)

### Laplacian of a pair $K \subset L$

The Laplacian of a pair $\mathbf{L}_k(K \subset L)$ is defined

\begin{equation}
    \mathbf{L}_k(K \subset L) = \mathbf{L}_k^{UP}(K \subset L) + \mathbf{L}_k^{DOWN}(K),
\end{equation}

where
- $\mathbf{L}_k^{DOWN}(K) = \mathbf{B}_k(K) \mathbf{B}_k(K)^T$,  
- $\mathbf{L}_k^{UP}(K \subset L) = \mathbf{L}_k^{UP}(L)~/~\mathbf{L}_k^{UP}(L \setminus K)$.

$\mathbf{L}_k^{UP}(K \subset L)$ is the Schur complement of $\mathbf{L}_k^{UP}(L \setminus K)$ in $\mathbf{L}_k^{UP}(L) = \mathbf{B}_{k+1}(L) \mathbf{B}_{k+1}(L)^T$

Algorithm, compute:
- $\mathbf{L}_k^{DOWN}(K)$,
- $\mathbf{L}_k^{UP}(L)$, from it compute
- $\mathbf{L}_k^{UP}(K \subset L)$ as the Schur complement of $\mathbf{L}_k^{UP}(L \setminus K)$ in $\mathbf{L}_k^{UP}(L)$, finally
- $\mathbf{L}_k(K \subset L) = \mathbf{L}_k^{UP}(K \subset L) + \mathbf{L}_k^{DOWN}(K)$.

#### Triangle

Let $K = \{0, 1, 2, 01, 02, 12\}$ and $L = \{0, 1, 2, 01, 02, 12, 012\}$.

In [ ]:
simplices = [[1, 2], [1, 3], [2, 3]]

K = SimplicialComplex()

for simplex in simplices:
    K.add(simplex)

B1_K = K.boundary_operator_matrix(k=1).astype(int)

B1_K

In [ ]:
simplices = [[1, 2], [1, 3], [2, 3], [1, 2, 3]]

L = SimplicialComplex()

for simplex in simplices:
    L.add(simplex)

B1_L = L.boundary_operator_matrix(k=1).astype(int)
B2_L = L.boundary_operator_matrix(k=2).astype(int)

B1_L, B2_L

In [ ]:
L1_K_low = B1_K.T @ B1_K
L1_K_low

In [ ]:
L1_L_up = B2_L @ B2_L.T
L1_L_up

#### Persistent 1-Laplacian of the inclusion $K \subset L$

1-dim Laplacian of $K \subset L$ is the sum of the upper and lower parts

\begin{align}
    \mathbf{L}_1(K \subset L) &= \mathbf{L}_1^{UP}(K \subset L) + \mathbf{L}_1^{DOWN}(K),\\
                              &= \mathbf{L}_1^{UP}(L) + \mathbf{L}_1^{DOWN}(K)
\end{align}

For a pair $K \subset L$, representing triangle filtration, $\mathbf{L}_1^{UP}(K \subset L)$ coincide with $\mathbf{L}_1^{UP}(L)$ as the set of 1-simplices is the same.

In [ ]:
L1_LK = L1_L_up + L1_K_low
L1_LK

#### Persistent Betti-1 number of the inclusion $K \subset L$,  $\beta_1(K \subset L)$

Persistent Betti-1 number of the inclusion $K \subset L$, $\beta_1(K \subset L)$ is 0, as the cycle of $K$ is filled in $L$.

In [ ]:
np.linalg.eigvalsh(L1_LK)

### Truss

Given a pair of simplicial complexes $K \subset L$, such that
- $K = \{1, 2, 3, 4, 5, 12, 13, 23, 24, 34, 35, 45, 123 \}$, and  
- $L = \{ 1, 2, 3, 4, 5, 6, 12, 13, 23, 24, 34, 35, 45, 46, 56, 123, 234 \}$

compute 1-dim Betti numbers of $K$ and $L$, along with 1-dim persistent Betti number of $K \subset L$.

In [ ]:
simplices = [[1, 2], [1, 3], [2, 3], [2, 4], [3, 4], [3, 5], [4, 5], [1, 2, 3]]

K = SimplicialComplex()

for simplex in simplices:
    K.add(simplex)

B1_K = K.boundary_operator_matrix(k=1).astype(int)
B2_K = K.boundary_operator_matrix(k=2).astype(int)

B1_K, B2_K

In [ ]:
simplices = [[1, 2], [1, 3], [2, 3], [2, 4], [3, 4], [3, 5], [4, 5], [4, 6], [5, 6], [1, 2, 3], [2, 3, 4]]

L = SimplicialComplex()

for simplex in simplices:
    L.add(simplex)

B1_L = L.boundary_operator_matrix(k=1).astype(int)
B2_L = L.boundary_operator_matrix(k=2).astype(int)

B1_L, B2_L

#### Betti-1 number of $K$, $\beta_1(K)$

1-dim Laplacian of $K$ is the sum of the upper and lower parts, using $B_2(K): C_2(K) \rightarrow C_1(K)$ and $B_2(K): C_2(K) \rightarrow C_1(K)$

\begin{equation}
    \mathbf{L}_1(K) = \mathbf{B}_2(K) \mathbf{B}_2(K)^T + \mathbf{B}_1(K)^T \mathbf{B}_1(K)
\end{equation}

In [ ]:
L1_K_up = B2_K @ B2_K.T
L1_K_up

In [ ]:
L1_K_down = B1_K.T @ B1_K
L1_K_down

In [ ]:
L1_K = L1_K_up + L1_K_down
L1_K

1-dim Betti number of $K$ is the dimension of $\ker \mathbf{L}_1(K) = 2$, as there are 2 cycles in $K$.

In [ ]:
np.linalg.eigvalsh(L1_K)

#### Betti-1 number of $L$, $\beta_1(L)$

1-dim Laplacian of $L$ is the sum of the upper and lower parts, using $B_2(L): C_2(L) \rightarrow C_1(L)$ and $B_2(K): C_2(L) \rightarrow C_1(L)$

\begin{equation}
    \mathbf{L}_1(L) = \mathbf{B}_2(L) \mathbf{B}_2(L)^T + \mathbf{B}_1(L)^T \mathbf{B}_1(L)
\end{equation}

In [ ]:
L1_L_up = B2_L @ B2_L.T
L1_L_up

In [ ]:
L1_K_down = B1_L.T @ B1_L
L1_K_down

In [ ]:
L1_L = L1_L_up + L1_K_down
L1_L

1-dim Betti number of $K$ is the dimension of $\ker \mathbf{L}_1(K) = 2$, as there are 2 cycles in $K$, yet on different vertices that ones of $K$.

In [ ]:
np.linalg.eigvalsh(L1_L)

#### Persistent Betti-1 number of an inclusion $K \subset L$, $\beta_1(K \subset L)$

1-dim Laplacian of $K \subset L$ is the sum of the upper and lower parts, using $B_2(K): C_2(K) \rightarrow C_1(K)$ and $B_2(K): C_2(K) \rightarrow C_1(K)$

\begin{equation}
    \mathbf{L}_1(K \subset L) = \mathbf{B}_2(K \subset L) \mathbf{B}_2(K \subset L)^T + \mathbf{B}_1(K)^T \mathbf{B}_1(K)
\end{equation}

In [ ]:
L1_K_down = B1_K.T @ B1_K
L1_K_down

In [ ]:
B2_L_to_K = B2_L[:-2]
B2_L_to_K

In [ ]:
L1_L_up = B2_L @ B2_L.T
L1_L_up

### Schur complement

Given a block matrix $\mathbf{M}$

\begin{equation}
    \mathbf{M} = \begin{pmatrix}
    \mathbf{A} & \mathbf{B}\\
    \mathbf{C} & \mathbf{D}
    \end{pmatrix}
\end{equation}

the Schur complement of $\mathbf{D}$ in $\mathbf{M}$ is defined

\begin{equation}
    \mathbf{M} / \mathbf{D} = \mathbf{A} - \mathbf{B} \mathbf{D}^{\dagger}\mathbf{C},
\end{equation}

where $\mathbf{D}^{\dagger}$ is the pseudoinverse of $\mathbf{D}$.

Analogously, the Schur complement of $\mathbf{A}$ in $\mathbf{M}$ is defined

\begin{equation}
    \mathbf{M} / \mathbf{A} = \mathbf{D} - \mathbf{B} \mathbf{A}^{\dagger}\mathbf{C},
\end{equation}

In [ ]:
A, B, D = schur_blocks(L1_L_up, np.arange(7), np.arange(7))
B2_K_in_L = (A - B @ np.linalg.pinv(D) @ B.T).astype(int)
B2_K_in_L

In [ ]:
L1_KL_up = B2_K_in_L @ B2_K_in_L.T
L1_KL_up

In [ ]:
L1_KL = L1_KL_up + L1_K_down
L1_KL

Persistent 1-Betti number of the inclusion $K \subset L$ is the dimension of $\ker \mathbf{L}_1(K \subset L) = 1$, as there a single cycle in $K$, that persists when included into $L$.

In [ ]:
np.linalg.eigvalsh(L1_KL)

### Simplicial Wattz-Strogatz model

In [ ]:
n, m, ps = 50, 10, [0.05, 0.075, 0.15]

fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16.5,5), dpi=200)

def stack(idx):
    ret = np.empty((0, 2))
    for _id in idx:
        ret = np.vstack((ret, X[_id,:]))
    return ret

for i, p_i in enumerate(ps):
    
    G = nx.generators.random_graphs.watts_strogatz_graph(n, m, p_i)
    K = expand_incremental(G, k=2)
    
    # boundary matrices
    B1 = boundary_matrix(K, k=1)
    B2 = boundary_matrix(K, k=2)
    
    # Laplacians
    L0 = B1 @ B1.T
    L1 = B1.T @ B1 + B2 @ B2.T
    
    # eigenvalues/vectors
    eigenvalues_v, eigenvectors_v = np.linalg.eigh(L0)
    eigenvalues_e, eigenvectors_e = np.linalg.eigh(L1)
    
    # average clustering coefficient/shortest path lengths of nodes
    average_clustering = nx.average_clustering(G)
    average_shortest_path_length = nx.average_shortest_path_length(G)
    
    X_dict = nx.kamada_kawai_layout(G)
    X = np.zeros((n,2))
    for j in X_dict:
        X[j] = X_dict[j]
    
    ax[i].set_title("p={}, $\Gamma$={:.2f}, $\Pi$={:.2f}".format(p_i, average_clustering, average_shortest_path_length))
    ax[i].scatter(X[:,0], X[:,1], c="k", s=10)

    # plot edges
    for e in G.edges():
        (start_id, end_id) = e
        ax[i].plot([X[start_id,0], X[end_id,0]], [X[start_id,1], X[end_id,1]], 'c-', alpha=0.5)
        
    # plot triangles
    for triangle in K[2]:
        t = plt.Polygon(stack(triangle), color="blue", alpha=0.02)
        ax[i].add_patch(t)

plt.show()

#### Higher-order Laplacian spectra

In [ ]:
N = 100

spectra0 = []
spectra012 = []
spectra1 = []
spectra0_norm = []
spectra012_norm = []
spectra1_norm = []

for i, p_i in enumerate(tqdm(np.repeat(ps, N))):
    
    # generate graph
    G = nx.generators.random_graphs.watts_strogatz_graph(n, m, p_i, seed=i)
    
    # clique complex of graph
    K = expand_incremental(G, k=2)
    
    # boundary matrices
    B1 = boundary_matrix(K, k=1)
    B2 = boundary_matrix(K, k=2)
    B22 = generalized_boundary_matrix(K, k=2, p=2)
    
    # Laplacians
    L0 = B1 @ B1.T
    L1 = B1.T @ B1 + B2 @ B2.T
    L012 = B22 @ B22.T
    
    # normalized Laplacians
    L0_norm = normalize(L0)
    L1_norm = normalize(L1)
    L012_norm = normalize(L012)
    
    # eigenvalues
    eigenvalues_v, _ = np.linalg.eigh(L0)
    eigenvalues_e, _ = np.linalg.eigh(L1)
    eigenvalues_vt, _ = np.linalg.eigh(L012)
    eigenvalues_v_norm = np.sort(np.linalg.eigvals(L0_norm))
    eigenvalues_e_norm = np.sort(np.linalg.eigvals(L1_norm))
    eigenvalues_vt_norm = np.sort(np.linalg.eigvals(L012_norm))
    
    spectra0.append(eigenvalues_v)
    spectra1.append(eigenvalues_e)
    spectra012.append(eigenvalues_vt)
    spectra0_norm.append(eigenvalues_v_norm)
    spectra1_norm.append(eigenvalues_e_norm)
    spectra012_norm.append(eigenvalues_vt_norm)
    
spectra0 = np.array(spectra0)
spectra1 = np.array(spectra1)
spectra012 = np.array(spectra012)
spectra0_norm = np.array(spectra0_norm)
spectra1_norm = np.array(spectra1_norm)
spectra012_norm = np.array(spectra012_norm)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16.5,5), dpi=200)
ax[0].set_title("$L_0$ spectrum")
ax[1].set_title("$L_{1}$ spectrum")
ax[2].set_title("$L_{012}$ spectrum")
ax[0].plot(spectra0[0:N].T, c="r", alpha=0.33)
ax[0].plot(spectra0[N:N*2].T, c="g", alpha=0.4)
ax[0].plot(spectra0[N*2:N*3].T, c="b", alpha=0.1)
ax[1].plot(spectra1[0:N].T, c="r", alpha=0.33)
ax[1].plot(spectra1[N:2*N].T, c="g", alpha=0.4)
ax[1].plot(spectra1[N*2:N*3].T, c="b", alpha=0.1)
ax[2].plot(spectra012[0:N].T, c="r", alpha=0.33)
ax[2].plot(spectra012[N:2*N].T, c="g", alpha=0.4)
ax[2].plot(spectra012[N*2:N*3].T, c="b", alpha=0.1)
plt.show()

### Classification

In [ ]:
n_repeats = 10
y = np.concatenate((np.ones(N) * 0, np.ones(N) * 1, np.ones(N) * 2))

#### L0

In [ ]:
%%time
results = []
for r in range(n_repeats):
    clf = RandomForestClassifier(random_state=r)
    results.append(list(cross_val_score(clf, spectra0, y, cv=5)))
    
results = np.array(results)
print("{:.2f}±{:.2f}".format(np.mean(results) * 100, np.std(results) * 100))

#### L1

In [ ]:
%%time
results = []
for r in range(n_repeats):
    clf = RandomForestClassifier(random_state=r)
    results.append(list(cross_val_score(clf, spectra1, y, cv=5)))
    
results = np.array(results)
print("{:.2f}±{:.2f}".format(np.mean(results) * 100, np.std(results) * 100))

#### L012

In [ ]:
%%time
results = []
for r in range(n_repeats):
    clf = RandomForestClassifier(random_state=r)
    results.append(list(cross_val_score(clf, spectra012, y, cv=5)))
    
results = np.array(results)
print("{:.2f}±{:.2f}".format(np.mean(results) * 100, np.std(results) * 100))